# BSH Amine Activity Prediction Model

Predict whether a BSH enzyme will be active on a given amine.

**Input**: Enzyme embedding (ProtT5, 1024-dim) + Amine fingerprint (Morgan, 1024-bit) = 2048-dim

**Output**: Binary classification (active/inactive)

**Models**: Random Forest, XGBoost, MLP

**Split Strategy**: Hold out enzymes (not random pairs) to test generalization to new enzymes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, 
    precision_score, recall_score, accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    balanced_accuracy_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# XGBoost
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    print("XGBoost not installed, will skip XGBoost model")
    HAS_XGB = False

# RDKit for Morgan fingerprints
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    HAS_RDKIT = True
except ImportError:
    print("RDKit not installed, will use random fingerprints as placeholder")
    HAS_RDKIT = False

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = Path("../models")

# Model-specific output directories
MODEL_OUTPUT_DIR = OUTPUT_DIR / "model_outputs"
RF_OUTPUT_DIR = MODEL_OUTPUT_DIR / "rf"
XGB_OUTPUT_DIR = MODEL_OUTPUT_DIR / "xgb"
MLP_OUTPUT_DIR = MODEL_OUTPUT_DIR / "mlp"
COMPARISON_OUTPUT_DIR = MODEL_OUTPUT_DIR / "comparison"

# Create directories
for d in [MODEL_DIR, MODEL_OUTPUT_DIR, RF_OUTPUT_DIR, XGB_OUTPUT_DIR, MLP_OUTPUT_DIR, COMPARISON_OUTPUT_DIR]:
    d.mkdir(exist_ok=True, parents=True)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Data

In [ ]:
# Load activity labels
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
print(f"Activity data: {df_activity.shape}")

# Filter: exclude controls and canonical amines (Approach 2)
controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']

df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]

print(f"After filtering: {df_activity.shape}")
print(f"Enzymes: {df_activity['Enzyme'].nunique()}")
print(f"Amines: {df_activity['Amine'].nunique()}")
print(f"\nClass distribution:")
print(df_activity['active_approach2'].value_counts())
print(f"\n% Active: {100*df_activity['active_approach2'].mean():.1f}%")

In [ ]:
# Load enzyme embeddings
df_embed = pd.read_csv(OUTPUT_DIR / "enzyme_embeddings.csv")
print(f"Embeddings: {df_embed.shape}")

# Extract embedding columns
embed_cols = [c for c in df_embed.columns if c.startswith('enz_')]
print(f"Embedding dimensions: {len(embed_cols)}")

# Create enzyme -> embedding mapping
enzyme_embeddings = {}
for _, row in df_embed.iterrows():
    enzyme_embeddings[row['Enzyme']] = row[embed_cols].values.astype(np.float32)

print(f"Enzymes with embeddings: {len(enzyme_embeddings)}")

In [ ]:
# Load amine SMILES and compute Morgan fingerprints
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")

# Create name mapping (normalize names)
name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',  # approximate
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

# Get unique amines from activity data
amines_needed = df_activity['Amine'].unique()
print(f"Amines needed: {len(amines_needed)}")

# Compute Morgan fingerprints
FP_BITS = 1024
amine_fingerprints = {}

if HAS_RDKIT:
    for _, row in df_smiles.iterrows():
        name = row['Compound_Name']
        smiles = row['SMILES']
        
        # Normalize name
        norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
        
        if pd.isna(smiles):
            continue
            
        # Parse SMILES (handle salts by taking first component)
        smiles_clean = smiles.split('.')[0]
        mol = Chem.MolFromSmiles(smiles_clean)
        
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=FP_BITS)
            amine_fingerprints[norm_name] = np.array(fp)
else:
    # Placeholder: random fingerprints
    for amine in amines_needed:
        amine_fingerprints[amine] = np.random.randint(0, 2, FP_BITS).astype(np.float32)

print(f"Fingerprints computed: {len(amine_fingerprints)}")

# Check coverage
missing = [a for a in amines_needed if a not in amine_fingerprints]
if missing:
    print(f"\nMissing fingerprints for: {missing}")
    # Use zero fingerprints for missing (e.g., 'unconjugated')
    for a in missing:
        amine_fingerprints[a] = np.zeros(FP_BITS, dtype=np.float32)

## 2. Prepare Training Data

In [ ]:
# Aggregate activity by (Enzyme, Amine) - use majority vote across products
# (multiple products per enzyme-amine pair, e.g., Di, Mono, Tri hydroxyl classes)

df_agg = df_activity.groupby(['Enzyme', 'Amine']).agg(
    active=('active_approach2', 'any'),  # active if ANY product is active
    n_products=('ProductName', 'count'),
    n_active_products=('active_approach2', 'sum')
).reset_index()

print(f"Aggregated data: {df_agg.shape}")
print(f"\nClass distribution after aggregation:")
print(df_agg['active'].value_counts())
print(f"\n% Active: {100*df_agg['active'].mean():.1f}%")

In [ ]:
# Build feature matrix
X_list = []
y_list = []
enzyme_list = []
amine_list = []

skipped = 0
for _, row in df_agg.iterrows():
    enzyme = row['Enzyme']
    amine = row['Amine']
    
    if enzyme not in enzyme_embeddings:
        skipped += 1
        continue
    if amine not in amine_fingerprints:
        skipped += 1
        continue
    
    # Concatenate enzyme embedding and amine fingerprint
    features = np.concatenate([
        enzyme_embeddings[enzyme],
        amine_fingerprints[amine]
    ])
    
    X_list.append(features)
    y_list.append(int(row['active']))
    enzyme_list.append(enzyme)
    amine_list.append(amine)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)

print(f"Feature matrix: {X.shape}")
print(f"Labels: {y.shape}")
print(f"Skipped: {skipped}")
print(f"\nClass balance: {np.bincount(y)}")
print(f"% Active: {100*y.mean():.1f}%")

## 3. Enzyme-Stratified Split

Hold out ~20% of enzymes for testing. Stratify by enzyme activity profile.

In [ ]:
# Get unique enzymes
enzymes = np.array(enzyme_list)
unique_enzymes = np.unique(enzymes)
print(f"Unique enzymes: {len(unique_enzymes)}")

# Compute activity profile for each enzyme (% active across amines)
enzyme_profiles = {}
for enz in unique_enzymes:
    mask = enzymes == enz
    enzyme_profiles[enz] = y[mask].mean()

# Cluster enzymes by activity profile for stratified split
profile_values = np.array([enzyme_profiles[e] for e in unique_enzymes]).reshape(-1, 1)

# Simple stratification: split into 5 bins based on activity level
bins = pd.cut(profile_values.flatten(), bins=5, labels=False)

# Split enzymes
train_enzymes, test_enzymes = train_test_split(
    unique_enzymes, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=bins
)

print(f"Train enzymes: {len(train_enzymes)}")
print(f"Test enzymes: {len(test_enzymes)}")

# Create masks
train_mask = np.isin(enzymes, train_enzymes)
test_mask = np.isin(enzymes, test_enzymes)

X_train_full = X[train_mask]
y_train_full = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]

print(f"\nTrain samples: {len(X_train_full)} ({100*len(X_train_full)/len(X):.1f}%)")
print(f"Test samples: {len(X_test)} ({100*len(X_test)/len(X):.1f}%)")

print(f"\nTrain class balance: {np.bincount(y_train_full)} ({100*y_train_full.mean():.1f}% active)")
print(f"Test class balance: {np.bincount(y_test)} ({100*y_test.mean():.1f}% active)")

In [ ]:
# Further split training data into train and validation
# Get training enzyme list
train_enzymes_arr = enzymes[train_mask]
unique_train_enzymes = np.unique(train_enzymes_arr)

# Stratify by activity profile
train_profile_values = np.array([enzyme_profiles[e] for e in unique_train_enzymes]).reshape(-1, 1)
train_bins = pd.cut(train_profile_values.flatten(), bins=5, labels=False)

train_enzymes_final, val_enzymes = train_test_split(
    unique_train_enzymes,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=train_bins
)

train_final_mask = np.isin(train_enzymes_arr, train_enzymes_final)
val_mask = np.isin(train_enzymes_arr, val_enzymes)

X_train = X_train_full[train_final_mask]
y_train = y_train_full[train_final_mask]
X_val = X_train_full[val_mask]
y_val = y_train_full[val_mask]

print(f"Final split:")
print(f"  Train: {len(X_train)} samples ({len(train_enzymes_final)} enzymes)")
print(f"  Val: {len(X_val)} samples ({len(val_enzymes)} enzymes)")
print(f"  Test: {len(X_test)} samples ({len(test_enzymes)} enzymes)")

In [ ]:
# Scale features for MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Compute class weights
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
class_weight = {0: 1.0, 1: n_neg / n_pos}
print(f"Class weights: {class_weight}")

## 4. Train Models

### 4.1 Random Forest

In [ ]:
%%time

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight=class_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    oob_score=True
)

rf.fit(X_train, y_train)

print(f"OOB Score: {rf.oob_score_:.3f}")
print(f"Train Accuracy: {rf.score(X_train, y_train):.3f}")
print(f"Val Accuracy: {rf.score(X_val, y_val):.3f}")

### 4.2 XGBoost

In [ ]:
%%time

if HAS_XGB:
    # XGBoost with early stopping
    scale_pos_weight = n_neg / n_pos
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        early_stopping_rounds=20,
        eval_metric='logloss',
        n_jobs=-1
    )
    
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    xgb_train_history = xgb_model.evals_result()
    best_iteration = xgb_model.best_iteration
    
    print(f"Best iteration: {best_iteration}")
    print(f"Train Accuracy: {xgb_model.score(X_train, y_train):.3f}")
    print(f"Val Accuracy: {xgb_model.score(X_val, y_val):.3f}")
else:
    xgb_model = None
    xgb_train_history = None
    print("XGBoost not available")

### 4.3 MLP

In [ ]:
%%time

# MLP with early stopping
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    alpha=0.01,  # L2 regularization
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,  # Internal validation for early stopping
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=True
)

mlp.fit(X_train_scaled, y_train)

print(f"\nEpochs trained: {mlp.n_iter_}")
print(f"Train Accuracy: {mlp.score(X_train_scaled, y_train):.3f}")
print(f"Val Accuracy: {mlp.score(X_val_scaled, y_val):.3f}")

## 5. Evaluate Models

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, is_scaled=False):
    """Compute all metrics for a model."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba)
    }
    
    return metrics, y_pred, y_proba

# Evaluate all models
results = []

# Random Forest
rf_metrics, rf_pred, rf_proba = evaluate_model(rf, X_test, y_test, 'Random Forest')
results.append(rf_metrics)

# XGBoost
if HAS_XGB and xgb_model is not None:
    xgb_metrics, xgb_pred, xgb_proba = evaluate_model(xgb_model, X_test, y_test, 'XGBoost')
    results.append(xgb_metrics)
else:
    xgb_metrics, xgb_pred, xgb_proba = None, None, None

# MLP
mlp_metrics, mlp_pred, mlp_proba = evaluate_model(mlp, X_test_scaled, y_test, 'MLP')
results.append(mlp_metrics)

# Display results
df_results = pd.DataFrame(results)
print("\n" + "="*60)
print("MODEL COMPARISON ON TEST SET")
print("="*60)
print(df_results.to_string(index=False))

## 6. Visualizations

In [ ]:
# Training curves for XGBoost
if HAS_XGB and xgb_train_history is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    train_loss = xgb_train_history['validation_0']['logloss']
    val_loss = xgb_train_history['validation_1']['logloss']
    
    ax.plot(train_loss, label='Train', linewidth=2)
    ax.plot(val_loss, label='Validation', linewidth=2)
    ax.axvline(best_iteration, color='red', linestyle='--', label=f'Best iteration ({best_iteration})')
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Log Loss', fontsize=12)
    ax.set_title('XGBoost Training Curve', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(XGB_OUTPUT_DIR / 'xgb_training_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {XGB_OUTPUT_DIR / 'xgb_training_curve.png'}")

In [ ]:
# MLP training curve
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(mlp.loss_curve_, label='Training Loss', linewidth=2)
if hasattr(mlp, 'validation_scores_') and mlp.validation_scores_ is not None:
    ax.plot(mlp.validation_scores_, label='Validation Score', linewidth=2)

ax.axvline(mlp.n_iter_, color='red', linestyle='--', alpha=0.7, label=f'Early stop ({mlp.n_iter_})')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('MLP Training Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MLP_OUTPUT_DIR / 'mlp_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {MLP_OUTPUT_DIR / 'mlp_training_curve.png'}")

In [ ]:
def plot_model_performance(model_name, y_test, y_pred, y_proba, feature_importance=None, feature_names=None, save_path=None):
    """Create performance plots for a single model."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # ROC curve
    ax = axes[0, 0]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curve', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Precision-Recall curve
    ax = axes[0, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    ax.plot(recall, precision, linewidth=2, label=f'PR (AUC = {pr_auc:.3f})')
    ax.axhline(y_test.mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
    ax.set_xlabel('Recall', fontsize=11)
    ax.set_ylabel('Precision', fontsize=11)
    ax.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Confusion matrix (using matplotlib instead of seaborn)
    ax = axes[1, 0]
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Inactive', 'Active'])
    ax.set_yticklabels(['Inactive', 'Active'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14,
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    ax.set_title('Confusion Matrix', fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    # Feature importance (if available)
    ax = axes[1, 1]
    if feature_importance is not None:
        # Top 20 features
        top_idx = np.argsort(feature_importance)[-20:][::-1]
        top_imp = feature_importance[top_idx]
        top_names = [feature_names[i] if feature_names else f'F{i}' for i in top_idx]
        
        colors = ['steelblue' if 'enz_' in str(n) else 'coral' for n in top_names]
        ax.barh(range(len(top_imp)), top_imp, color=colors)
        ax.set_yticks(range(len(top_imp)))
        ax.set_yticklabels(top_names, fontsize=8)
        ax.set_xlabel('Importance', fontsize=11)
        ax.set_title('Top 20 Feature Importance', fontsize=12, fontweight='bold')
        ax.invert_yaxis()
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='steelblue', label='Enzyme'),
                          Patch(facecolor='coral', label='Amine')]
        ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Feature importance\nnot available', 
                ha='center', va='center', fontsize=12, transform=ax.transAxes)
        ax.set_axis_off()
    
    fig.suptitle(f'{model_name} Performance', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()

# Feature names
feature_names = [f'enz_{i}' for i in range(1024)] + [f'fp_{i}' for i in range(1024)]

In [ ]:
# Random Forest performance
plot_model_performance(
    'Random Forest', y_test, rf_pred, rf_proba,
    feature_importance=rf.feature_importances_,
    feature_names=feature_names,
    save_path=RF_OUTPUT_DIR / 'rf_performance.png'
)

In [ ]:
# XGBoost performance
if HAS_XGB and xgb_model is not None:
    plot_model_performance(
        'XGBoost', y_test, xgb_pred, xgb_proba,
        feature_importance=xgb_model.feature_importances_,
        feature_names=feature_names,
        save_path=XGB_OUTPUT_DIR / 'xgb_performance.png'
    )

In [ ]:
# MLP performance (no feature importance)
plot_model_performance(
    'MLP', y_test, mlp_pred, mlp_proba,
    feature_importance=None,
    feature_names=None,
    save_path=MLP_OUTPUT_DIR / 'mlp_performance.png'
)

In [ ]:
# Model comparison plot
fig = plt.figure(figsize=(16, 10))

# Panel 1: Metrics comparison bar chart
ax1 = fig.add_subplot(2, 2, 1)
metrics_to_plot = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
x = np.arange(len(metrics_to_plot))
width = 0.25

models = df_results['model'].tolist()
colors = ['#3498db', '#2ecc71', '#e74c3c']

for i, (model, color) in enumerate(zip(models, colors[:len(models)])):
    values = [df_results[df_results['model'] == model][m].values[0] for m in metrics_to_plot]
    ax1.bar(x + i*width, values, width, label=model, color=color, edgecolor='black')

ax1.set_xlabel('Metric', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Model Comparison: All Metrics', fontsize=14, fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(['Acc', 'Bal Acc', 'Prec', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC'], fontsize=10)
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3, axis='y')

# Panel 2: Overlaid ROC curves
ax2 = fig.add_subplot(2, 2, 2)

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
ax2.plot(fpr_rf, tpr_rf, linewidth=2, color=colors[0], label=f'RF (AUC={rf_metrics["roc_auc"]:.3f})')

if xgb_proba is not None:
    fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)
    ax2.plot(fpr_xgb, tpr_xgb, linewidth=2, color=colors[1], label=f'XGB (AUC={xgb_metrics["roc_auc"]:.3f})')

fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp_proba)
ax2.plot(fpr_mlp, tpr_mlp, linewidth=2, color=colors[2], label=f'MLP (AUC={mlp_metrics["roc_auc"]:.3f})')

ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curves Comparison', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3)

# Panel 3: Overlaid PR curves
ax3 = fig.add_subplot(2, 2, 3)

prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_proba)
ax3.plot(rec_rf, prec_rf, linewidth=2, color=colors[0], label=f'RF (AUC={rf_metrics["pr_auc"]:.3f})')

if xgb_proba is not None:
    prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, xgb_proba)
    ax3.plot(rec_xgb, prec_xgb, linewidth=2, color=colors[1], label=f'XGB (AUC={xgb_metrics["pr_auc"]:.3f})')

prec_mlp, rec_mlp, _ = precision_recall_curve(y_test, mlp_proba)
ax3.plot(rec_mlp, prec_mlp, linewidth=2, color=colors[2], label=f'MLP (AUC={mlp_metrics["pr_auc"]:.3f})')

ax3.axhline(y_test.mean(), color='gray', linestyle='--', alpha=0.5, label='Baseline')
ax3.set_xlabel('Recall', fontsize=12)
ax3.set_ylabel('Precision', fontsize=12)
ax3.set_title('Precision-Recall Curves Comparison', fontsize=14, fontweight='bold')
ax3.legend(loc='lower left', fontsize=10)
ax3.grid(True, alpha=0.3)

# Panel 4: Summary table
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')

# Create summary table
table_data = df_results[['model', 'accuracy', 'f1', 'roc_auc', 'pr_auc']].copy()
table_data.columns = ['Model', 'Accuracy', 'F1', 'ROC-AUC', 'PR-AUC']
for col in ['Accuracy', 'F1', 'ROC-AUC', 'PR-AUC']:
    table_data[col] = table_data[col].apply(lambda x: f'{x:.3f}')

table = ax4.table(
    cellText=table_data.values,
    colLabels=table_data.columns,
    loc='center',
    cellLoc='center',
    colColours=['#f0f0f0']*5
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.5)

ax4.set_title('Summary Metrics', fontsize=14, fontweight='bold', y=0.8)

plt.tight_layout()
plt.savefig(COMPARISON_OUTPUT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMPARISON_OUTPUT_DIR / 'model_comparison.png'}")

## 7. Per-Amine Analysis

In [ ]:
# Get per-amine performance
test_enzymes_arr = enzymes[test_mask]
test_amines_arr = np.array(amine_list)[test_mask]

per_amine_results = []

for amine in np.unique(test_amines_arr):
    mask = test_amines_arr == amine
    if mask.sum() < 5:  # Skip amines with too few samples
        continue
    
    y_true = y_test[mask]
    
    # Skip if only one class
    if len(np.unique(y_true)) < 2:
        continue
    
    row = {'amine': amine, 'n_samples': mask.sum()}
    
    # RF
    y_pred_rf = rf_pred[mask]
    row['rf_f1'] = f1_score(y_true, y_pred_rf, zero_division=0)
    
    # XGBoost
    if xgb_pred is not None:
        y_pred_xgb = xgb_pred[mask]
        row['xgb_f1'] = f1_score(y_true, y_pred_xgb, zero_division=0)
    else:
        row['xgb_f1'] = np.nan
    
    # MLP
    y_pred_mlp = mlp_pred[mask]
    row['mlp_f1'] = f1_score(y_true, y_pred_mlp, zero_division=0)
    
    per_amine_results.append(row)

df_per_amine = pd.DataFrame(per_amine_results).sort_values('rf_f1', ascending=False)
print("Per-amine F1 scores:")
print(df_per_amine.to_string(index=False))

In [ ]:
# Plot per-amine performance
fig, ax = plt.subplots(figsize=(14, 6))

amines = df_per_amine['amine'].tolist()
x = np.arange(len(amines))
width = 0.25

ax.bar(x - width, df_per_amine['rf_f1'], width, label='Random Forest', color='#3498db', edgecolor='black')
if 'xgb_f1' in df_per_amine.columns and not df_per_amine['xgb_f1'].isna().all():
    ax.bar(x, df_per_amine['xgb_f1'], width, label='XGBoost', color='#2ecc71', edgecolor='black')
ax.bar(x + width, df_per_amine['mlp_f1'], width, label='MLP', color='#e74c3c', edgecolor='black')

ax.set_xlabel('Amine', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Amine F1 Score by Model', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(amines, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(COMPARISON_OUTPUT_DIR / 'per_amine_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {COMPARISON_OUTPUT_DIR / 'per_amine_performance.png'}")

## 8. Save Models and Results

In [ ]:
# Save models
with open(MODEL_DIR / 'bsh_rf_classifier.pkl', 'wb') as f:
    pickle.dump(rf, f)
print(f"Saved: {MODEL_DIR / 'bsh_rf_classifier.pkl'}")

if HAS_XGB and xgb_model is not None:
    with open(MODEL_DIR / 'bsh_xgb_classifier.pkl', 'wb') as f:
        pickle.dump(xgb_model, f)
    print(f"Saved: {MODEL_DIR / 'bsh_xgb_classifier.pkl'}")

with open(MODEL_DIR / 'bsh_mlp_classifier.pkl', 'wb') as f:
    pickle.dump({'model': mlp, 'scaler': scaler}, f)
print(f"Saved: {MODEL_DIR / 'bsh_mlp_classifier.pkl'}")

In [ ]:
# Save training data
df_training_data = pd.DataFrame({
    'enzyme': enzyme_list,
    'amine': amine_list,
    'label': y,
    'split': ['train' if e in train_enzymes_final else 'val' if e in val_enzymes else 'test' for e in enzyme_list]
})
df_training_data.to_csv(MODEL_OUTPUT_DIR / 'model_training_data.csv', index=False)
print(f"Saved: {MODEL_OUTPUT_DIR / 'model_training_data.csv'}")

In [ ]:
# Save test predictions
df_predictions = pd.DataFrame({
    'enzyme': test_enzymes_arr,
    'amine': test_amines_arr,
    'y_true': y_test,
    'rf_pred': rf_pred,
    'rf_proba': rf_proba,
    'mlp_pred': mlp_pred,
    'mlp_proba': mlp_proba
})

if xgb_pred is not None:
    df_predictions['xgb_pred'] = xgb_pred
    df_predictions['xgb_proba'] = xgb_proba

df_predictions.to_csv(MODEL_OUTPUT_DIR / 'model_results.csv', index=False)
print(f"Saved: {MODEL_OUTPUT_DIR / 'model_results.csv'}")

In [ ]:
# Save metrics summary
df_results.to_csv(MODEL_OUTPUT_DIR / 'model_metrics_summary.csv', index=False)
print(f"Saved: {MODEL_OUTPUT_DIR / 'model_metrics_summary.csv'}")

## 9. Summary

In [ ]:
print("="*60)
print("BSH AMINE ACTIVITY PREDICTION - SUMMARY")
print("="*60)

print(f"\nData:")
print(f"  Total samples: {len(X):,}")
print(f"  Enzymes: {len(unique_enzymes)}")
print(f"  Amines: {len(np.unique(amine_list))}")
print(f"  Features: {X.shape[1]} (1024 enzyme + 1024 amine)")

print(f"\nSplit (enzyme hold-out):")
print(f"  Train: {len(X_train)} samples ({len(train_enzymes_final)} enzymes)")
print(f"  Val: {len(X_val)} samples ({len(val_enzymes)} enzymes)")
print(f"  Test: {len(X_test)} samples ({len(test_enzymes)} enzymes)")

print(f"\nBest Model:")
best_model = df_results.loc[df_results['pr_auc'].idxmax()]
print(f"  {best_model['model']} (PR-AUC = {best_model['pr_auc']:.3f})")

print(f"\nAll Models:")
for _, row in df_results.iterrows():
    print(f"  {row['model']:15s} - ROC-AUC: {row['roc_auc']:.3f}, PR-AUC: {row['pr_auc']:.3f}, F1: {row['f1']:.3f}")

print(f"\nFiles saved:")
print(f"  Models:")
print(f"    - {MODEL_DIR / 'bsh_rf_classifier.pkl'}")
print(f"    - {MODEL_DIR / 'bsh_xgb_classifier.pkl'}")
print(f"    - {MODEL_DIR / 'bsh_mlp_classifier.pkl'}")
print(f"  Outputs:")
print(f"    - {RF_OUTPUT_DIR / 'rf_performance.png'}")
print(f"    - {XGB_OUTPUT_DIR / 'xgb_performance.png'}")
print(f"    - {XGB_OUTPUT_DIR / 'xgb_training_curve.png'}")
print(f"    - {MLP_OUTPUT_DIR / 'mlp_performance.png'}")
print(f"    - {MLP_OUTPUT_DIR / 'mlp_training_curve.png'}")
print(f"    - {COMPARISON_OUTPUT_DIR / 'model_comparison.png'}")
print(f"    - {COMPARISON_OUTPUT_DIR / 'per_amine_performance.png'}")
print(f"    - {MODEL_OUTPUT_DIR / 'model_results.csv'}")
print(f"    - {MODEL_OUTPUT_DIR / 'model_metrics_summary.csv'}")